In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install transformers datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 23.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 17.9 MB/s eta 0:00:00


In [3]:
!pip install accelerate -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 9.1 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-m

#Load and Prepare the dataset

In [4]:
import pandas as pd
from datasets import Dataset

# Define paths to the .src and .tgt files
src_file = '/content/drive/MyDrive/codeT5p_data_@/train_data/train.src'
tgt_file = '/content/drive/MyDrive/codeT5p_data_@/train_data/train.tgt'

# Read the files into pandas DataFrames
with open(src_file, 'r', encoding='utf-8') as f:
    src_lines = f.readlines()

with open(tgt_file, 'r', encoding='utf-8') as f:
    tgt_lines = f.readlines()

# Ensure both files have the same number of lines
assert len(src_lines) == len(tgt_lines), "Source and target files must have the same number of lines."

# Create a DataFrame
data = {'input_text': src_lines, 'target_text': tgt_lines}
df = pd.DataFrame(data)

# Convert the DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(df)

#Change configuration of codeT5p

In [5]:
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer, T5Config, TrainingArguments, Trainer,AutoModel, AutoTokenizer
from datasets import load_dataset

# Load original configuration and tokenizer
model_name = "Salesforce/codet5p-220m"
config = T5Config.from_pretrained(model_name)
pretrained_model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/446M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

In [6]:
# Modify the n_positions parameter
config.n_positions = 2048  # new desired sequence length

# Reinitialize the model with the new configuration
model = T5ForConditionalGeneration(config)

# Modify the positional embeddings
model.shared = torch.nn.Embedding(config.vocab_size, config.d_model)
model.encoder.embed_tokens = model.shared
model.encoder.block[0].layer[0].SelfAttention.relative_attention_bias = torch.nn.Embedding(config.relative_attention_num_buckets, config.num_heads)
model.decoder.embed_tokens = model.shared
model.decoder.block[0].layer[0].SelfAttention.relative_attention_bias = torch.nn.Embedding(config.relative_attention_num_buckets, config.num_heads)


In [7]:
# Check if the configuration of the model is changed

# 1. Check the n_positions configuration
print(f"New n_positions: {model.config.n_positions}")

# 2. Check the embedding dimensions
print(f"Shared embedding shape: {model.shared.weight.shape}")
print(f"Encoder embed tokens shape: {model.encoder.embed_tokens.weight.shape}")
print(f"Decoder embed tokens shape: {model.decoder.embed_tokens.weight.shape}")

# 3. Check the relative attention bias embeddings
print(f"Encoder relative attention bias embedding shape: {model.encoder.block[0].layer[0].SelfAttention.relative_attention_bias.weight.shape}")
print(f"Decoder relative attention bias embedding shape: {model.decoder.block[0].layer[0].SelfAttention.relative_attention_bias.weight.shape}")


New n_positions: 1024
Shared embedding shape: torch.Size([32100, 768])
Encoder embed tokens shape: torch.Size([32100, 768])
Decoder embed tokens shape: torch.Size([32100, 768])
Encoder relative attention bias embedding shape: torch.Size([32, 12])
Decoder relative attention bias embedding shape: torch.Size([32, 12])


#Tokenize the Dataset

In [8]:
# Tokenize the dataset
def preprocess_function(examples):
    inputs = [ex.strip() for ex in examples['input_text']]
    targets = [ex.strip() for ex in examples['target_text']]
    model_inputs = tokenizer(inputs, max_length=2048, truncation=True, padding="max_length", return_tensors="pt")

    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=2048, truncation=True, padding="max_length",return_tensors="pt")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(
            preprocess_function,
            batched=True,
            remove_columns=dataset.column_names,
            num_proc=1,
            load_from_cache_file=False,
       )

# Remove 'input_text' field and set the format to PyTorch tensors
#tokfrom google.colab import driveenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Print a single example to check the structure
print("Single Example from Tokenized Dataset:")
print(tokenized_dataset[0])

Map:   0%|          | 0/1375 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:3946: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Single Example from Tokenized Dataset:
{'input_ids': [1, 2978, 1241, 67, 1153, 261, 269, 513, 67, 830, 79, 16, 269, 513, 67, 16114, 16, 269, 513, 67, 1937, 16, 269, 513, 67, 8734, 16, 269, 513, 67, 20390, 16, 269, 513, 67, 1672, 16, 269, 290, 16, 269, 761, 16, 269, 1022, 337, 30752, 21, 16, 269, 1022, 337, 30752, 21, 67, 438, 67, 90, 1236, 16, 269, 1022, 337, 30752, 22, 16, 269, 1022, 337, 30752, 22, 67, 438, 67, 90, 1236, 16, 269, 1022, 337, 30752, 23, 16, 269, 1022, 337, 30752, 23, 67, 438, 67, 90, 1236, 16, 269, 513, 67, 2463, 269, 11272, 269, 1569, 513, 67, 882, 67, 2556, 81, 67, 2019, 21, 273, 890, 14271, 21, 31, 269, 1569, 513, 67, 882, 67, 2556, 81, 67, 2019, 22, 273, 890, 14271, 22, 31, 269, 1569, 513, 67, 882, 67, 2556, 81, 67, 2019, 23, 273, 890, 14271, 24, 31, 269, 810, 513, 67, 830, 79, 31, 269, 810, 513, 67, 16114, 31, 269, 810, 513, 67, 1937, 31, 269, 876, 513, 67, 8734, 31, 269, 876, 513, 67, 20390, 31, 269, 876, 513, 67, 1672, 31, 269, 810, 306, 6938, 30, 20, 65, 290, 3

In [9]:
# Accessing the attributes of the first sample
first_sample = tokenized_dataset[0]

# Get the input_ids, attention_mask, and labels
input_ids = first_sample['input_ids']
attention_mask = first_sample['attention_mask']
labels = first_sample['labels']

# Check the lengths
input_ids_length = len(input_ids)
attention_mask_length = len(attention_mask)
labels_length = len(labels)

print(f"Input IDs Length: {input_ids_length}")
print(f"Attention Mask Length: {attention_mask_length}")
print(f"Labels Length: {labels_length}")


Input IDs Length: 1024
Attention Mask Length: 1024
Labels Length: 1024


In [10]:
# Loop through the dataset and print lengths for multiple samples
for i in range(5):  # Print lengths for the first 5 samples
    sample = tokenized_dataset[i]
    input_ids_length = len(sample['input_ids'])
    attention_mask_length = len(sample['attention_mask'])
    labels_length = len(sample['labels'])
    print(f"Sample {i+1}:")
    print(f"Input IDs Length: {input_ids_length}")
    print(f"Attention Mask Length: {attention_mask_length}")
    print(f"Labels Length: {labels_length}")
    print()


Sample 1:
Input IDs Length: 1024
Attention Mask Length: 1024
Labels Length: 1024

Sample 2:
Input IDs Length: 1024
Attention Mask Length: 1024
Labels Length: 1024

Sample 3:
Input IDs Length: 1024
Attention Mask Length: 1024
Labels Length: 1024

Sample 4:
Input IDs Length: 1024
Attention Mask Length: 1024
Labels Length: 1024

Sample 5:
Input IDs Length: 1024
Attention Mask Length: 1024
Labels Length: 1024



In [11]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=pretrained_model, padding=True)

In [12]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1375
})

In [13]:
# View the first 5 samples
print(tokenized_dataset[:5])


{'input_ids': [[1, 2978, 1241, 67, 1153, 261, 269, 513, 67, 830, 79, 16, 269, 513, 67, 16114, 16, 269, 513, 67, 1937, 16, 269, 513, 67, 8734, 16, 269, 513, 67, 20390, 16, 269, 513, 67, 1672, 16, 269, 290, 16, 269, 761, 16, 269, 1022, 337, 30752, 21, 16, 269, 1022, 337, 30752, 21, 67, 438, 67, 90, 1236, 16, 269, 1022, 337, 30752, 22, 16, 269, 1022, 337, 30752, 22, 67, 438, 67, 90, 1236, 16, 269, 1022, 337, 30752, 23, 16, 269, 1022, 337, 30752, 23, 67, 438, 67, 90, 1236, 16, 269, 513, 67, 2463, 269, 11272, 269, 1569, 513, 67, 882, 67, 2556, 81, 67, 2019, 21, 273, 890, 14271, 21, 31, 269, 1569, 513, 67, 882, 67, 2556, 81, 67, 2019, 22, 273, 890, 14271, 22, 31, 269, 1569, 513, 67, 882, 67, 2556, 81, 67, 2019, 23, 273, 890, 14271, 24, 31, 269, 810, 513, 67, 830, 79, 31, 269, 810, 513, 67, 16114, 31, 269, 810, 513, 67, 1937, 31, 269, 876, 513, 67, 8734, 31, 269, 876, 513, 67, 20390, 31, 269, 876, 513, 67, 1672, 31, 269, 810, 306, 6938, 30, 20, 65, 290, 31, 269, 810, 306, 6938, 30, 20, 65, 76

#Fine-tuning on custom data

In [14]:
# Training arguments
training_args = TrainingArguments(
    report_to='tensorboard',
    output_dir="/content/drive/MyDrive/n_position_RevEngg",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.05,
    warmup_steps=200,
    logging_dir="./logs",
    logging_first_step=True,
    logging_steps=10,
    save_total_limit=1,
    dataloader_drop_last=True,
    dataloader_num_workers=1,
    fp16=False,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Train the model
trainer.train()

# Save the final model
model.save_pretrained("./final_checkpoint")
tokenizer.save_pretrained("./final_checkpoint")

print('Finished training and saved the model to ./final_checkpoint')

/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Step,Training Loss
1,11.876300
10,10.298300
20,3.509600
30,2.080700
40,1.490500
50,1.125200
60,0.799200
70,0.724400
80,0.657200
90,0.635000


/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Finished training and saved the model to ./final_checkpoint
